© 2025 Mobile Perception Systems Lab at TU/e. All rights reserved. Licensed under the MIT License.

## Setup

In [ ]:
## Setup
# Dependencies are listed in requirements.txt at the repo root.
# From the repo root: pip install -r requirements.txt
import os
print(os.getcwd())  # should be the eomt/ directory


In [ ]:
import contextlib
import yaml
from lightning import seed_everything
import torch
from torch.nn import functional as F
from torch.amp.autocast_mode import autocast
import matplotlib.pyplot as plt
import numpy as np
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import RepositoryNotFoundError
import warnings
import importlib

seed_everything(0, verbose=False)

# Auto-detect GPU; falls back to CPU if no CUDA is available
device = "cuda:0" if torch.cuda.is_available() else "cpu"
device_type = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

BASE      = os.environ.get("MASKARCH_DATA_ROOT", "../data")  # see data/README.md


CITYSCAPES_DATA_PATH = BASE + "/datasets/cityscapes"



CS_CHECKPOINT_PATH   = BASE + "/checkpoints/cityscapes/eomt_cityscapes.bin"

img_idx = 0


def create_mapping(images, ignore_index):
    unique_ids = np.unique(np.concatenate([np.unique(img) for img in images]))
    valid_ids = unique_ids[unique_ids != ignore_index]
    colors = np.array(
        [plt.cm.hsv(i / len(valid_ids))[:3] for i in range(len(valid_ids))]
    )
    mapping = {cid: colors[i] for i, cid in enumerate(valid_ids)}
    mapping[ignore_index] = np.array([0, 0, 0])
    return mapping


def apply_colormap(image, mapping):
    colored_image = np.zeros((*image.shape, 3))
    for cid in np.unique(image):
        colored_image[image == cid] = mapping.get(cid, [0, 0, 0])
    return colored_image


def autocast_ctx():
    """float16 autocast on CUDA; no-op on CPU (float16 unsupported on CPU)."""
    if device_type == "cuda":
        return autocast(dtype=torch.float16, device_type="cuda")
    return contextlib.nullcontext()


## Part A – Cityscapes Semantic Model

Loads the EoMT model trained on **Cityscapes** for semantic segmentation (19 classes).
Produces a pixel-wise class label map.

### Load Cityscapes Dataset

Ensure the Cityscapes zip files (`leftImg8bit_trainvaltest.zip` and `gtFine_trainvaltest.zip`) are placed in `CITYSCAPES_DATA_PATH`.

In [ ]:
cs_config_path = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
with open(cs_config_path, "r") as f:
    cs_config = yaml.safe_load(f)

data_module_name, class_name = cs_config["data"]["class_path"].rsplit(".", 1)
data_module_cls = getattr(importlib.import_module(data_module_name), class_name)
data_module_kwargs = cs_config["data"].get("init_args", {})

cs_data = data_module_cls(
    path=CITYSCAPES_DATA_PATH,
    batch_size=1,
    num_workers=0,
    check_empty_targets=False,
    **data_module_kwargs
).setup()

print(f"Cityscapes val: {len(cs_data.val_dataloader().dataset)} images, "
      f"{cs_data.num_classes} classes, img_size={cs_data.img_size}")


### Build and Load Model Weights

Weights are downloaded automatically from the Hugging Face Hub (`tue-mps/cityscapes_semantic_eomt_base_640`).

In [ ]:
import os

warnings.filterwarnings(
    "ignore",
    message=r".*Attribute 'network' is an instance of `nn\.Module` and is already saved during checkpointing.*",
)


def build_model(config, num_classes, img_size):
    """Build EoMT model from config, move to device, return (model, lit_cls, model_kw)."""
    encoder_cfg = config["model"]["init_args"]["network"]["init_args"]["encoder"]
    enc_mod, enc_name = encoder_cfg["class_path"].rsplit(".", 1)
    encoder = getattr(importlib.import_module(enc_mod), enc_name)(
        img_size=img_size, **encoder_cfg.get("init_args", {})
    )

    net_cfg = config["model"]["init_args"]["network"]
    net_mod, net_name = net_cfg["class_path"].rsplit(".", 1)
    net_kw = {k: v for k, v in net_cfg["init_args"].items() if k != "encoder"}
    network = getattr(importlib.import_module(net_mod), net_name)(
        masked_attn_enabled=False, num_classes=num_classes, encoder=encoder, **net_kw
    )

    lit_mod, lit_name = config["model"]["class_path"].rsplit(".", 1)
    lit_cls = getattr(importlib.import_module(lit_mod), lit_name)
    model_kw = {k: v for k, v in config["model"]["init_args"].items() if k != "network"}
    if "stuff_classes" in config["data"].get("init_args", {}):
        model_kw["stuff_classes"] = config["data"]["init_args"]["stuff_classes"]

    model = lit_cls(img_size=img_size, num_classes=num_classes, network=network, **model_kw)
    return model.eval().to(device), lit_cls, model_kw


def load_weights(model, config, lit_cls, model_kw, num_classes, img_size, local_ckpt=None):
    """
    Load weights into model.
    Priority: local_ckpt (if the path exists) > HuggingFace Hub.
    """
    # 1. Try local checkpoint first
    if local_ckpt and os.path.isfile(local_ckpt):
        state_dict = torch.load(local_ckpt, map_location=device, weights_only=True)
        model.load_state_dict(state_dict, strict=False)
        print(f"Loaded local weights: {local_ckpt}")
        return model

    # 2. Fall back to HuggingFace Hub
    name = config.get("trainer", {}).get("logger", {}).get("init_args", {}).get("name")
    if name is None:
        warnings.warn("No logger name in config; skipping weight download.")
        return model
    try:
        ckpt_path = hf_hub_download(repo_id=f"tue-mps/{name}", filename="pytorch_model.bin")
        is_dinov3 = "dinov3" in name
        if is_dinov3:
            model_kw["ckpt_path"] = ckpt_path
            model_kw["delta_weights"] = True
            encoder_cfg = config["model"]["init_args"]["network"]["init_args"]["encoder"]
            enc_mod, enc_name = encoder_cfg["class_path"].rsplit(".", 1)
            encoder = getattr(importlib.import_module(enc_mod), enc_name)(
                img_size=img_size, **encoder_cfg.get("init_args", {})
            )
            net_cfg = config["model"]["init_args"]["network"]
            net_mod, net_name_inner = net_cfg["class_path"].rsplit(".", 1)
            net_kw = {k: v for k, v in net_cfg["init_args"].items() if k != "encoder"}
            network = getattr(importlib.import_module(net_mod), net_name_inner)(
                masked_attn_enabled=False, num_classes=num_classes, encoder=encoder, **net_kw
            )
            model = lit_cls(
                img_size=img_size, num_classes=num_classes, network=network, **model_kw
            ).eval().to(device)
        else:
            state_dict = torch.load(ckpt_path, map_location=device, weights_only=True)
            model.load_state_dict(state_dict, strict=False)
        print(f"Loaded HuggingFace weights: tue-mps/{name}")
    except RepositoryNotFoundError:
        warnings.warn(f"HF repo not found for {name}. Using random weights.")
    return model


# Build and load Cityscapes semantic model
# Uses CS_CHECKPOINT_PATH if the file exists, otherwise downloads from HuggingFace
cs_model, cs_lit_cls, cs_model_kw = build_model(cs_config, cs_data.num_classes, cs_data.img_size)
cs_model = load_weights(
    cs_model, cs_config, cs_lit_cls, cs_model_kw,
    cs_data.num_classes, cs_data.img_size,
    local_ckpt=CS_CHECKPOINT_PATH,
)


### Semantic Inference (pixel-wise classification)

Per-pixel class scores: $\sum_i p_i(c) \cdot m_i[h,w]$, then argmax over classes.

> Also works on a panoptic-trained model.

In [ ]:
#We didn't push the notebook output for the photo because due to metadata complexity GitHub marks the notebook as invalid — just run the cell to generate it locally.
IGNORE_INDEX = 255


def infer_semantic(img, target, model, data):
    with torch.no_grad(), autocast_ctx():
        imgs = [img.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]
        crops, origins = model.window_imgs_semantic(imgs)
        mask_logits_per_layer, class_logits_per_layer = model(crops)
        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], data.img_size, mode="bilinear"
        )
        crop_logits = model.to_per_pixel_logits_semantic(
            mask_logits, class_logits_per_layer[-1]
        )
        logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)
        preds = logits[0].argmax(0).cpu()

    pred_array = preds.numpy()
    target_array = model.to_per_pixel_targets_semantic([target], IGNORE_INDEX)[0].numpy()
    return pred_array, target_array


def plot_semantic_results(img, pred_array, target_array, suptitle=""):
    mapping = create_mapping([pred_array, target_array], IGNORE_INDEX)
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(img.permute(1, 2, 0).cpu().numpy())
    axes[0].set_title("Input Image")
    axes[1].imshow(apply_colormap(pred_array, mapping))
    axes[1].set_title("Semantic Prediction (19 Cityscapes classes)")
    axes[2].imshow(apply_colormap(target_array, mapping))
    axes[2].set_title("Ground Truth")
    for ax in axes:
        ax.axis("off")
    if suptitle:
        fig.suptitle(suptitle, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()


img, target = cs_data.val_dataloader().dataset[img_idx]
cs_pred_array, cs_target_array = infer_semantic(img, target, cs_model, cs_data)
plot_semantic_results(
    img, cs_pred_array, cs_target_array,
    suptitle="Part A – Cityscapes Semantic Model (EoMT trained on Cityscapes)"
)


## Part B – COCO Panoptic Model

Loads the EoMT model trained on **COCO** for panoptic segmentation (133 classes: 80 things + 53 stuff).
We run it on the **same Cityscapes validation image** from Part A to compare the two models visually.

> No COCO dataset files are required – the image is reused from Part A.

### Build and Load COCO Panoptic Model

Weights downloaded from `tue-mps/coco_panoptic_eomt_base_640`.

In [ ]:
coco_config_path = "configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml"
with open(coco_config_path, "r") as f:
    coco_config = yaml.safe_load(f)

COCO_IMG_SIZE = (640, 640)
COCO_NUM_CLASSES = 133
COCO_CHECKPOINT_PATH = BASE + "/checkpoints/coco/eomt_coco.bin"

coco_model, coco_lit_cls, coco_model_kw = build_model(
    coco_config, COCO_NUM_CLASSES, COCO_IMG_SIZE
)
coco_model = load_weights(
    coco_model, coco_config, coco_lit_cls, coco_model_kw, COCO_NUM_CLASSES, COCO_IMG_SIZE,
    local_ckpt=COCO_CHECKPOINT_PATH
)


### Panoptic Inference

Each pixel is assigned to the query $i$ maximising $p_i(c_i) \cdot m_i[h,w]$.
Stuff classes are merged; thing instances are kept distinct (black borders = instance boundaries).

Because the COCO model uses a **different class vocabulary** (133 COCO classes vs. 19 Cityscapes classes), the colours are unrelated to those in Part A.

In [ ]:
def draw_black_border(sem, inst, mapping):
    h, w = sem.shape
    out = np.zeros((h, w, 3))
    for s in np.unique(sem):
        out[sem == s] = mapping.get(s, [0, 0, 0])
    combined = sem.astype(np.int64) * 100000 + inst.astype(np.int64)
    border = np.zeros((h, w), dtype=bool)
    border[1:, :] |= combined[1:, :] != combined[:-1, :]
    border[:-1, :] |= combined[1:, :] != combined[:-1, :]
    border[:, 1:] |= combined[:, 1:] != combined[:, :-1]
    border[:, :-1] |= combined[:, 1:] != combined[:, :-1]
    out[border] = 0
    return out


def plot_panoptic_results(img, sem_pred, inst_pred, num_classes, suptitle=""):
    all_ids = np.unique(sem_pred)
    mapping = {
        s: [0, 0, 0] if s in (-1, num_classes)
        else list(plt.cm.hsv(i / max(len(all_ids), 1))[:3])
        for i, s in enumerate(all_ids)
    }
    vis_pred = draw_black_border(sem_pred, inst_pred, mapping)
    img_np = img.cpu().numpy().transpose(1, 2, 0)

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    axes[0].imshow(img_np)
    axes[0].set_title("Input Image")
    axes[1].imshow(vis_pred)
    axes[1].set_title(
        "Panoptic Prediction (133 COCO classes)\nblack borders = instance boundaries"
    )
    for ax in axes:
        ax.axis("off")
    if suptitle:
        fig.suptitle(suptitle, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()


def infer_panoptic_no_gt(img, model, mask_thresh=0.3, overlap_thresh=0.5):
    with torch.no_grad(), autocast_ctx():
        imgs = [img.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]
        transformed_imgs = model.resize_and_pad_imgs_instance_panoptic(imgs)
        mask_logits_per_layer, class_logits_per_layer = model(transformed_imgs)
        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], model.img_size, mode="bilinear"
        )
        mask_logits = model.revert_resize_and_pad_logits_instance_panoptic(
            mask_logits, img_sizes
        )
        preds = model.to_per_pixel_preds_panoptic(
            mask_logits,
            class_logits_per_layer[-1],
            model.stuff_classes,
            mask_thresh,
            overlap_thresh,
        )[0].cpu()

    pred = preds.numpy()
    return pred[..., 0], pred[..., 1]


coco_sem_pred, coco_inst_pred = infer_panoptic_no_gt(img, coco_model)
print(f"Valori unici in sem_pred: {np.unique(coco_sem_pred)}")

plot_panoptic_results(
    img, coco_sem_pred, coco_inst_pred, coco_model.num_classes,
    suptitle="Part B – COCO Panoptic Model (EoMT trained on COCO) – Cityscapes image"
)


In [ ]:
# ======== STEP 4: fair mIoU comparison (EoMT-COCO vs EoMT-Cityscapes) ========
# Run from the eomt/ directory.
!python pip_eval.py
